## Logistic Regression For Sentiment Analysis of Reviews

### Imports

In [1]:
import re
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

### Load Dataset

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

# convert label
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})
df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


### Train Test Split 80-20

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,     
    random_state=42,    
    stratify=df["label"] 
)

In [4]:
train_df.head()

,review,sentiment,label
47808,I caught this little gem totally by accident b...,positive,1
20154,I can't believe that I let myself into this mo...,negative,0
43069,*spoiler alert!* it just gets to me the nerve ...,negative,0
19413,If there's one thing I've learnt from watching...,negative,0
13673,"I remember when this was in theaters, reviews ...",negative,0


In [5]:
#Convert to List
train_data = list(zip(train_df["review"], train_df["label"]))
test_data = list(zip(test_df["review"], test_df["label"]))
train_data[0]

('I caught this little gem totally by accident back in 1980 or \'81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn\'t stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it\'s even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! A definite 10!',
 1)

### Text Preprocessing

In [ ]:
STOPWORDS = {
    "the","a","an","is","it","in","on","at","to","for","of","and","or",
    "but","i","was","this","that","with","as","be","have","had","he",
    "she","they","we","are","were","been","has","do","did","not","by",
    "from","its","my","our","your","their","so","if","about","which",
    "would","could","should","more","also","just","than","then","when",
    "there","what","all","can","will","one","into","up","out","no","get"
}

def clean_text(text):
    # remove html tags
    text  = re.sub(r'<[^>]+>', ' ', text)     
    # only letters  
    text  = re.sub(r"[^a-zA-Z']", ' ', text)    
    words = text.lower().split()
    # just store clean and needed words only
    words = [w for w in words if w not in STOPWORDS and len(w) > 1]
    return words

### n-grams

In [7]:
def get_ngrams(words):
    bigrams = [words[i] + "_" + words[i+1] for i in range(len(words) - 1)]
    return words + bigrams

### Build Vocabulary

In [8]:
def build_vocab(data, max_features=15000, min_freq=3, max_doc_ratio=0.9):
    word_count = {}
    doc_freq   = {}
    total_docs = len(data)

    for text, _ in data:
        words        = get_ngrams(clean_text(text))
        unique_words = set(words)

        for word in words:
            word_count[word] = word_count.get(word, 0) + 1
        for word in unique_words:
            doc_freq[word] = doc_freq.get(word, 0) + 1

    filtered  = {
        word: count for word, count in word_count.items()
        if count >= min_freq
        and doc_freq.get(word, 0) / total_docs <= max_doc_ratio
    }

    top_words = sorted(filtered, key=filtered.get, reverse=True)[:max_features]
    return {word: index for index, word in enumerate(top_words)}

vocab = build_vocab(train_data)
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 15000


### IDF

In [9]:
def compute_idf(data, vocab):
    total_docs = len(data)
    doc_freq   = np.zeros(len(vocab))

    for text, _ in data:
        unique_words = set(get_ngrams(clean_text(text)))
        for word in unique_words:
            if word in vocab:
                doc_freq[vocab[word]] += 1

    return np.log((total_docs + 1) / (doc_freq + 1)) + 1

idf = compute_idf(train_data, vocab)

### Vectorization

In [10]:
def text_to_vector(text, vocab, idf):
    words  = get_ngrams(clean_text(text))
    vector = np.zeros(len(vocab))

    for word in words:
        if word in vocab:
            vector[vocab[word]] += 1

    if vector.sum() > 0:
        vector = vector / vector.sum()          

    vector = vector * idf                        

    norm = np.sqrt((vector ** 2).sum())          
    if norm > 0:
        vector = vector / norm

    return vector

### X Y Train and Test

In [11]:
X_train = np.array([text_to_vector(text, vocab, idf) for text, _ in train_data])
y_train = np.array([label for _, label in train_data], dtype=np.float64)

X_test  = np.array([text_to_vector(text, vocab, idf) for text, _ in test_data])
y_test  = np.array([label for _, label in test_data], dtype=np.float64)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")


X_train shape: (40000, 15000)
X_test  shape: (10000, 15000)


### Logistic Regression

In [12]:
class LogisticRegression:

    def __init__(self, lr=0.5, epochs=30, l2=0.001, batch_size=64):
        self.lr         = lr
        self.epochs     = epochs
        self.l2         = l2
        self.batch_size = batch_size
        self.weights    = None
        self.bias       = 0.0

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        num_samples, num_features = X.shape
        self.weights   = np.zeros(num_features)
        self.bias      = 0.0
        sample_indices = list(range(num_samples))

        for epoch in range(self.epochs):
            current_lr = self.lr / (1 + 0.05 * epoch)   # lr decay
            np.random.shuffle(sample_indices)
            total_loss = 0.0

            for start in range(0, num_samples, self.batch_size):
                batch   = sample_indices[start : start + self.batch_size]
                X_batch = X[batch]
                y_batch = y[batch]

                predictions     = self.sigmoid(X_batch.dot(self.weights) + self.bias)
                errors          = predictions - y_batch
                weight_gradient = X_batch.T.dot(errors) / len(batch)
                bias_gradient   = errors.mean()

                self.weights -= current_lr * (weight_gradient + self.l2 * self.weights)
                self.bias    -= current_lr * bias_gradient

                predictions = np.clip(predictions, 1e-15, 1 - 1e-15)
                total_loss += (-y_batch * np.log(predictions) - (1 - y_batch) * np.log(1 - predictions)).sum()

            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:2d}/{self.epochs}  |  loss: {total_loss/num_samples:.4f}  |  lr: {current_lr:.4f}")

    def predict(self, X):
        probabilities = self.sigmoid(X.dot(self.weights) + self.bias)
        return (probabilities >= 0.5).astype(int)

    def predict_review(self, review_text):
        vector = text_to_vector(review_text, vocab, idf)
        vector = vector.reshape(1, -1)    
        prob   = self.sigmoid(vector.dot(self.weights) + self.bias)[0]
        label  = "POSITIVE" if prob >= 0.5 else "NEGATIVE"
        print(f"\nReview    : {review_text[:80]}...")
        print(f"Sentiment : {label}")
        print(f"Confidence: {prob:.4f}  (closer to 1.0 = positive, 0.0 = negative)")


### Model Training

In [13]:
print("\nTraining...")
model = LogisticRegression(lr=0.5, epochs=30)
model.fit(X_train, y_train)


Training...
  Epoch  5/30  |  loss: 0.5562  |  lr: 0.4167
  Epoch 10/30  |  loss: 0.5402  |  lr: 0.3448
  Epoch 15/30  |  loss: 0.5372  |  lr: 0.2941
  Epoch 20/30  |  loss: 0.5365  |  lr: 0.2564
  Epoch 25/30  |  loss: 0.5362  |  lr: 0.2273
  Epoch 30/30  |  loss: 0.5361  |  lr: 0.2041


In [14]:
import pickle

saved_model = {
    "weights": model.weights,
    "bias": model.bias,
    "vocab": vocab,
    "idf": idf
}

with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(saved_model, f)

print("Model saved successfully.")

Model saved successfully.


### Predictions and Accuracy RESULTS

In [15]:
print("\n── Results ──")
y_pred = model.predict(X_test)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\n  True  Negatives  (correctly predicted negative) : {cm[0][0]}")
print(f"  False Positives  (negative predicted as positive): {cm[0][1]}")
print(f"  False Negatives  (positive predicted as negative): {cm[1][0]}")
print(f"  True  Positives  (correctly predicted positive)  : {cm[1][1]}")


── Results ──
Accuracy  : 0.8552
Precision : 0.8272
Recall    : 0.8980
F1 Score  : 0.8611

Confusion Matrix:
[[4062  938]
 [ 510 4490]]

  True  Negatives  (correctly predicted negative) : 4062
  False Positives  (negative predicted as positive): 938
  False Negatives  (positive predicted as negative): 510
  True  Positives  (correctly predicted positive)  : 4490


### Unseen Data Test

In [22]:
# model.predict_review("Very good work")
# model.predict_review("Terrible job and rude behaviour")
# model.predict_review("Not bad and not good, he was ok and fine")
# model.predict_review("Professionally done, great work")
# model.predict_review("Quickly Completed , surely was experienced")
# model.predict_review("Very bad work, didn't expect this")
# model.predict_review("Great worked, helped me a lot")
# model.predict_review("Well done , could have done better")
# model.predict_review("Very Professional work, good job, well done!")

model.predict_review("Professionally done, great work, Nice and Best")
model.predict_review("Terrible, rude, not nice, unprofessional, bad")


Review    : Professionally done, great work, Nice and Best...
Sentiment : POSITIVE
Confidence: 0.8166  (closer to 1.0 = positive, 0.0 = negative)

Review    : Terrible, rude, not nice, unprofessional, bad...
Sentiment : NEGATIVE
Confidence: 0.2821  (closer to 1.0 = positive, 0.0 = negative)
